# Introduction to Data Visualization

**Primary outcome:** Create and interpret 6 core chart types using Matplotlib, and apply chart selection principles to real Titanic data.

**Time:** ~45–90 minutes

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain *why* visualization matters — and what makes it more effective than raw tables
2. Build **bar charts**, **histograms**, **scatter plots**, **line charts**, **pie/donut charts**, and **subplot dashboards** using `matplotlib`
3. Customize every chart: titles, axis labels, colors, figure size, and layout
4. Apply a **chart selection guide** to choose the right chart type for any data question

---

## Why Does Visualization Matter?

Data visualization turns numbers into insight. The human brain processes visual information far faster than text or tables — a well-designed chart can reveal patterns that would take minutes (or hours) to spot in a spreadsheet.

### Pre-attentive Attributes

Our visual system detects certain properties **before conscious thought** — in under 250ms. These are called **pre-attentive attributes**:

| Attribute | Example use |
|-----------|-------------|
| **Color** | Highlight survived vs. not survived |
| **Length / Height** | Bar chart values |
| **Position** | Scatter plot coordinates |
| **Size** | Bubble chart magnitude |
| **Shape** | Category markers in a scatter plot |

Good charts exploit these attributes to make the key message instantly visible.

## 0. Setup — Imports & Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns          # used only to load the dataset

plt.style.use('seaborn-v0_8-whitegrid')   # clean grid background
np.random.seed(42)

# Load the Titanic dataset
df = sns.load_dataset('titanic')

print('Dataset loaded! Shape:', df.shape)
print('Columns:', df.columns.tolist())

## 1. Why Visualization? — Table vs. Chart

The same data presented as a table versus a bar chart tells a very different story. Let's compare them side by side.

We'll look at **survival rate by passenger class** — a simple three-row table that becomes immediately clear as a chart.

In [ ]:
# Compute survival rate per class
survival_by_class = df.groupby('pclass')['survived'].mean().reset_index()
survival_by_class.columns = ['Class', 'Survival Rate']
survival_by_class['Survival Rate'] = (survival_by_class['Survival Rate'] * 100).round(1)

# --- Side-by-side comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: render the table as text
axes[0].axis('off')
table_data = [['Class', 'Survival Rate (%)']]
for _, row in survival_by_class.iterrows():
    table_data.append([f"Class {int(row['Class'])}", f"{row['Survival Rate']}%"])

tbl = axes[0].table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    loc='center',
    cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(13)
tbl.scale(1.4, 2.0)
axes[0].set_title('Raw Table', fontsize=14, fontweight='bold', pad=20)

# Right: bar chart
colors = ['#2ecc71', '#f39c12', '#e74c3c']
bars = axes[1].bar(
    [f'Class {c}' for c in survival_by_class['Class']],
    survival_by_class['Survival Rate'],
    color=colors,
    edgecolor='white',
    linewidth=1.2
)
# Add value labels on bars
for bar, val in zip(bars, survival_by_class['Survival Rate']):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val}%',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )
axes[1].set_title('Bar Chart', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Survival Rate (%)', fontsize=12)
axes[1].set_ylim(0, 80)

fig.suptitle('Same Data — Table vs. Chart', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print()
print('Key insight: The chart instantly shows Class 1 passengers survived at twice the rate of Class 3.')
print('You have to read every row of the table to reach the same conclusion.')

## 2. Dataset: Titanic

The **Titanic dataset** contains information about 891 passengers aboard the RMS Titanic, which sank in April 1912. It is one of the most widely used datasets in data science.

| Column | Description |
|--------|-------------|
| `survived` | 0 = did not survive, 1 = survived |
| `pclass` | Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd |
| `sex` | Passenger sex |
| `age` | Age in years |
| `sibsp` | # of siblings / spouses aboard |
| `parch` | # of parents / children aboard |
| `fare` | Ticket fare (£) |
| `embarked` | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |

Let's explore it.

In [ ]:
# First few rows
print('=== First 5 rows ===')
df.head()

In [ ]:
# Column types and missing values
print('=== Data Types & Missing Values ===')
df.info()

In [ ]:
# Summary statistics for numeric columns
print('=== Summary Statistics ===')
df.describe().round(2)

## 3. Bar Charts: Comparing Categories

**When to use a bar chart:**
- Comparing values across a small number of categories (2–10)
- Showing counts, sums, or averages per group
- Ranking items (sort the bars for extra clarity)

**Avoid** bar charts when you have more than ~15 categories (use a horizontal bar chart or table instead) or when you want to show distributions (use a histogram).

We'll create two bar charts:
1. Survival rate by passenger class — grouped bars
2. Survival count by sex

In [ ]:
# --- Chart 3a: Survival rate by class (grouped bars: survived vs. not survived) ---

# Compute counts
class_survival = df.groupby(['pclass', 'survived']).size().unstack(fill_value=0)
class_survival.columns = ['Did Not Survive', 'Survived']

classes = [1, 2, 3]
x = np.arange(len(classes))    # label positions
width = 0.35                   # width of each bar

fig, ax = plt.subplots(figsize=(9, 5))

bars1 = ax.bar(x - width/2, class_survival['Did Not Survive'], width,
               label='Did Not Survive', color='#e74c3c', edgecolor='white')
bars2 = ax.bar(x + width/2, class_survival['Survived'], width,
               label='Survived', color='#2ecc71', edgecolor='white')

# Labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=10)

ax.set_title('Titanic Survival Count by Passenger Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Passenger Class', fontsize=12)
ax.set_ylabel('Number of Passengers', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(['Class 1', 'Class 2', 'Class 3'])
ax.legend(fontsize=11)
ax.set_ylim(0, 380)

plt.tight_layout()
plt.show()

print('Class 3 had the most passengers but also the lowest survival count.')

In [ ]:
# --- Chart 3b: Survival count by sex ---

sex_survival = df.groupby(['sex', 'survived']).size().unstack(fill_value=0)
sex_survival.columns = ['Did Not Survive', 'Survived']

fig, ax = plt.subplots(figsize=(7, 5))

x = np.arange(len(sex_survival.index))
width = 0.35

ax.bar(x - width/2, sex_survival['Did Not Survive'], width,
       label='Did Not Survive', color='#e74c3c', edgecolor='white')
ax.bar(x + width/2, sex_survival['Survived'], width,
       label='Survived', color='#2ecc71', edgecolor='white')

ax.set_title('Titanic Survival Count by Sex', fontsize=14, fontweight='bold')
ax.set_xlabel('Sex', fontsize=12)
ax.set_ylabel('Number of Passengers', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(['Female', 'Male'])
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print('Female passengers had a dramatically higher survival rate than male passengers.')

## 4. Histograms & Distributions

**When to use a histogram:**
- Showing the **distribution** of a single continuous variable
- Identifying skew, outliers, or multi-modal patterns
- Understanding data spread before modeling

**Key parameter — `bins`:** Too few bins hides detail; too many creates noise. A good starting point is the square root of the number of rows, or experiment with 10, 20, and 50 to see what reveals the most structure.

We'll plot the distribution of `age` and `fare`, then show how bin count changes the picture.

In [ ]:
# --- Chart 4a: Age distribution ---

age_clean = df['age'].dropna()

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(age_clean, bins=30, color='#3498db', edgecolor='white', linewidth=0.6)

ax.axvline(age_clean.mean(), color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Mean age: {age_clean.mean():.1f}')
ax.axvline(age_clean.median(), color='#f39c12', linestyle='--', linewidth=2,
           label=f'Median age: {age_clean.median():.1f}')

ax.set_title('Age Distribution of Titanic Passengers', fontsize=14, fontweight='bold')
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f'Age range: {age_clean.min():.0f} – {age_clean.max():.0f} years')
print(f'Missing age values: {df["age"].isna().sum()} out of {len(df)}')

In [ ]:
# --- Chart 4b: Fare distribution (right-skewed) ---

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df['fare'], bins=40, color='#9b59b6', edgecolor='white', linewidth=0.6)

ax.axvline(df['fare'].mean(), color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Mean fare: £{df["fare"].mean():.1f}')
ax.axvline(df['fare'].median(), color='#f39c12', linestyle='--', linewidth=2,
           label=f'Median fare: £{df["fare"].median():.1f}')

ax.set_title('Fare Distribution of Titanic Passengers', fontsize=14, fontweight='bold')
ax.set_xlabel('Fare (£)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print('Fare is heavily right-skewed — most passengers paid under £50,')
print('but a few paid over £500, pulling the mean far above the median.')

In [ ]:
# --- Chart 4c: Effect of bin count on age histogram ---

bin_counts = [5, 20, 50]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, bins in zip(axes, bin_counts):
    ax.hist(age_clean, bins=bins, color='#3498db', edgecolor='white', linewidth=0.5)
    ax.set_title(f'bins = {bins}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Age (years)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)

fig.suptitle('Effect of Bin Count on Age Histogram', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('bins=5 : Too coarse — hides the distribution shape.')
print('bins=20: A good balance — reveals the peak and slight right skew.')
print('bins=50: Too fine — noisy, individual bars too thin to read.')

## 5. Scatter Plots: Relationships

**When to use a scatter plot:**
- Exploring the relationship (correlation) between two continuous variables
- Spotting clusters, outliers, or trends
- Adding a third dimension via color or size

**Correlation vs. Causation:**
> A scatter plot can show that two variables move together (correlation), but it **cannot prove one causes the other**. For example, higher fares correlate with survival — but that's because wealthy passengers in Class 1 had better access to lifeboats, not because money itself saved them.

We'll plot **age vs. fare**, colored by whether the passenger survived.

In [ ]:
# --- Chart 5: Age vs. Fare, colored by survival ---

# Drop rows with missing age
scatter_df = df.dropna(subset=['age'])

fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    scatter_df['age'],
    scatter_df['fare'],
    c=scatter_df['survived'],          # color encodes survival (0 or 1)
    cmap='RdYlGn',                     # red = not survived, green = survived
    alpha=0.6,
    edgecolors='none',
    s=40                               # point size
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Survived (0 = No, 1 = Yes)', fontsize=11)
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Did Not Survive', 'Survived'])

ax.set_title('Age vs. Fare — Colored by Survival', fontsize=14, fontweight='bold')
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Fare (£)', fontsize=12)

plt.tight_layout()
plt.show()

print('Passengers who paid higher fares (top of chart) skew more green (survived).')
print('The very bottom of the chart (low fare) is mostly red — Class 3 passengers.')

## 6. Line Charts: Trends Over Time (or Ordered Groups)

**When to use a line chart:**
- Showing how a value changes **over time** (or along an ordered axis)
- Highlighting trends, cycles, or turning points
- Comparing multiple trends on the same axis

**Lines vs. Bars for ordered categories:**
- Use **bars** when the categories are discrete and the gaps between them are not meaningful (e.g., individual stores)
- Use **lines** when the ordering matters and you want to emphasise *direction of change* (e.g., age groups, years)

We don't have a time column in the Titanic dataset, so we'll use **age groups** as our ordered axis — showing how survival rate changes across age brackets.

In [ ]:
# --- Chart 6: Survival rate by age group ---

age_df = df.dropna(subset=['age']).copy()

# Create 10-year bins: 0-9, 10-19, 20-29, ...
bins = list(range(0, 91, 10))
labels = [f'{b}–{b+9}' for b in bins[:-1]]
age_df['age_group'] = pd.cut(age_df['age'], bins=bins, labels=labels, right=False)

# Survival rate per age group
survival_by_age = age_df.groupby('age_group', observed=True)['survived'].agg(['mean', 'count'])
survival_by_age.columns = ['survival_rate', 'count']
survival_by_age['survival_pct'] = (survival_by_age['survival_rate'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(11, 5))

# Line: survival rate
ax1.plot(
    range(len(survival_by_age)),
    survival_by_age['survival_pct'],
    color='#2ecc71', linewidth=2.5, marker='o', markersize=7,
    label='Survival Rate (%)'
)

# Annotate each point
for i, (_, row) in enumerate(survival_by_age.iterrows()):
    ax1.annotate(
        f'{row["survival_pct"]}%',
        xy=(i, row['survival_pct']),
        xytext=(0, 10), textcoords='offset points',
        ha='center', fontsize=9, color='#27ae60'
    )

# Secondary axis: passenger count per group (bar)
ax2 = ax1.twinx()
ax2.bar(
    range(len(survival_by_age)),
    survival_by_age['count'],
    color='#3498db', alpha=0.3, label='Passenger Count'
)
ax2.set_ylabel('Passenger Count', fontsize=11, color='#3498db')
ax2.tick_params(axis='y', labelcolor='#3498db')

ax1.set_title('Survival Rate & Passenger Count by Age Group', fontsize=14, fontweight='bold')
ax1.set_xlabel('Age Group', fontsize=12)
ax1.set_ylabel('Survival Rate (%)', fontsize=12)
ax1.set_xticks(range(len(survival_by_age)))
ax1.set_xticklabels(survival_by_age.index, rotation=30)
ax1.set_ylim(0, 100)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print('Children (0–9) had the highest survival rate (~59%).')
print('Working-age men (20–49) made up most passengers and had lower survival rates.')

## 7. Pie & Donut Charts

**When to use a pie chart:**
- Showing part-to-whole composition for **3–5 categories**
- When the exact percentages matter less than the visual proportions

**When NOT to use a pie chart:**
- More than 5–6 slices — the small slices become illegible
- When you want to compare values precisely (use a bar chart instead)
- When slices are close in size — the human eye cannot accurately judge small angle differences

We'll look at the **embarkation port breakdown**, then convert it to a donut chart — a slight variation that adds a label area in the center.

In [ ]:
# --- Chart 7: Embarkation port — Pie & Donut side by side ---

port_counts = df['embarked'].value_counts()
port_labels = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
labels = [port_labels.get(p, p) for p in port_counts.index]
colors = ['#3498db', '#e67e22', '#2ecc71']
explode = (0.03, 0.03, 0.03)          # slight gap between slices

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# --- Pie chart ---
axes[0].pie(
    port_counts,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    textprops={'fontsize': 12}
)
axes[0].set_title('Pie Chart:\nEmbarkation Port', fontsize=13, fontweight='bold')

# --- Donut chart (wedge with white centre hole) ---
wedges, texts, autotexts = axes[1].pie(
    port_counts,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    textprops={'fontsize': 12},
    wedgeprops={'width': 0.55}         # creates the donut hole
)
# Centre label
axes[1].text(0, 0, f'n = {port_counts.sum()}', ha='center', va='center',
             fontsize=13, fontweight='bold', color='#2c3e50')
axes[1].set_title('Donut Chart:\nEmbarkation Port', fontsize=13, fontweight='bold')

fig.suptitle('Where Did Passengers Board?', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('72% of passengers boarded at Southampton (England).')
print('Note: 2 passengers have a missing embarkation port (excluded from chart).')

## 8. Subplots: Dashboard Layout

`plt.subplots(rows, cols)` lets you arrange multiple charts into a single figure — useful for dashboards, reports, or comparing several views at once.

We'll combine four of our charts into a 2×2 summary dashboard.

In [ ]:
# --- Chart 8: 2x2 Dashboard ---

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Titanic Data — Summary Dashboard', fontsize=18, fontweight='bold', y=1.01)

# ---------- Top-left: Survival by class (bar) ----------
ax = axes[0, 0]
surv_rate = df.groupby('pclass')['survived'].mean() * 100
bar_colors = ['#f1c40f', '#95a5a6', '#cd7f32']
ax.bar([f'Class {c}' for c in surv_rate.index], surv_rate.values,
       color=bar_colors, edgecolor='white')
ax.set_title('Survival Rate by Class', fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Survival Rate (%)')
ax.set_ylim(0, 80)
for i, v in enumerate(surv_rate.values):
    ax.text(i, v + 1.5, f'{v:.0f}%', ha='center', fontsize=10, fontweight='bold')

# ---------- Top-right: Age distribution (histogram) ----------
ax = axes[0, 1]
ax.hist(df['age'].dropna(), bins=25, color='#3498db', edgecolor='white', linewidth=0.5)
ax.axvline(df['age'].median(), color='#e74c3c', linestyle='--', linewidth=1.8,
           label=f'Median: {df["age"].median():.0f}')
ax.set_title('Age Distribution', fontweight='bold')
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)

# ---------- Bottom-left: Age vs. Fare scatter ----------
ax = axes[1, 0]
sc_df = df.dropna(subset=['age'])
scatter = ax.scatter(sc_df['age'], sc_df['fare'],
                     c=sc_df['survived'], cmap='RdYlGn',
                     alpha=0.55, s=25, edgecolors='none')
fig.colorbar(scatter, ax=ax).set_label('Survived', fontsize=9)
ax.set_title('Age vs. Fare (colored by survival)', fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Fare (£)')

# ---------- Bottom-right: Embarkation donut ----------
ax = axes[1, 1]
port_counts = df['embarked'].value_counts()
port_labels_map = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
labels_pie = [port_labels_map.get(p, p) for p in port_counts.index]
ax.pie(port_counts, labels=labels_pie, autopct='%1.0f%%',
       colors=['#3498db', '#e67e22', '#2ecc71'],
       startangle=90, textprops={'fontsize': 10},
       wedgeprops={'width': 0.55})
ax.text(0, 0, 'Port', ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_title('Embarkation Port', fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Chart Selection Guide

Not sure which chart to use? Start here.

| Question you're asking | Data type | Recommended chart |
|------------------------|-----------|-------------------|
| How do categories compare? | Category + number | **Bar chart** |
| How is this variable distributed? | Single continuous | **Histogram** |
| Is there a relationship between two variables? | Two continuous | **Scatter plot** |
| How does a value change over time / ordered groups? | Time + number | **Line chart** |
| What share does each category contribute? (≤5 groups) | Category + proportion | **Pie / Donut chart** |
| How does distribution differ across groups? | Category + continuous | **Box plot** |
| What is the relationship between 3+ variables? | Multi-variable | **Bubble / heatmap** |

### Quick decision tree

```
Start
 ├─ Comparing categories? ──────────────────── Bar chart
 ├─ Single variable, continuous? ──────────── Histogram
 ├─ Two continuous variables? ─────────────── Scatter plot
 ├─ Change over time / ordered? ───────────── Line chart
 └─ Part-to-whole? (few categories)
        ├─ ≤ 5 groups ──────────────────────── Pie / Donut
        └─ > 5 groups ──────────────────────── Stacked bar
```

### Golden rules

1. **Always label** your axes and add a title
2. **Choose colors with purpose** — use color to encode meaning, not decoration
3. **Less is more** — remove chartjunk (3-D effects, excessive gridlines, decorative borders)
4. **Start bar charts at zero** — truncating the y-axis exaggerates differences
5. **Order matters** — sort bars by value when there is no natural order

## 10. Practice Exercises

Work through each exercise below. Write your code in the blank cell provided. Hints and solutions are folded underneath each exercise.

---

### Exercise 1 — Bar Chart

Create a **horizontal bar chart** showing the **average fare paid by each passenger class**. Include a title, axis labels, and value annotations on each bar.

*Hint: use `ax.barh()` for horizontal bars. Flip `xlabel` and `ylabel`.*

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 1</strong> (click to expand)</summary>

```python
avg_fare = df.groupby('pclass')['fare'].mean()

fig, ax = plt.subplots(figsize=(8, 4))

bars = ax.barh(
    [f'Class {c}' for c in avg_fare.index],
    avg_fare.values,
    color=['#f1c40f', '#95a5a6', '#cd7f32'],
    edgecolor='white'
)

for bar, val in zip(bars, avg_fare.values):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f'£{val:.1f}',
        va='center', fontsize=11, fontweight='bold'
    )

ax.set_title('Average Fare by Passenger Class', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Fare (£)', fontsize=12)
ax.set_ylabel('Passenger Class', fontsize=12)
ax.set_xlim(0, 130)

plt.tight_layout()
plt.show()
```
</details>

---

### Exercise 2 — Histogram

Plot **overlapping histograms** of `age` for passengers who survived vs. those who did not. Use `alpha=0.6` so both are visible. Add a legend, title, and axis labels.

*Hint: call `plt.hist()` twice on the same axes — once for each group.*

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 2</strong> (click to expand)</summary>

```python
survived_ages = df[df['survived'] == 1]['age'].dropna()
died_ages     = df[df['survived'] == 0]['age'].dropna()

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(died_ages, bins=25, alpha=0.6, color='#e74c3c',
        edgecolor='white', label='Did Not Survive')
ax.hist(survived_ages, bins=25, alpha=0.6, color='#2ecc71',
        edgecolor='white', label='Survived')

ax.set_title('Age Distribution: Survived vs. Did Not Survive', fontsize=14, fontweight='bold')
ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()
```
</details>

---

### Exercise 3 — Subplots Dashboard

Create a **1×2 subplot** showing:
- Left: a pie chart of passenger class distribution (how many passengers in each class)
- Right: a bar chart of average age per class

Use `fig.suptitle()` for an overall title and `plt.tight_layout()` before showing.

In [ ]:
# Your code here


<details>
<summary><strong>Solution — Exercise 3</strong> (click to expand)</summary>

```python
class_counts = df['pclass'].value_counts().sort_index()
avg_age_class = df.groupby('pclass')['age'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Passenger Class Overview', fontsize=15, fontweight='bold')

# Left: Pie
axes[0].pie(
    class_counts,
    labels=[f'Class {c}' for c in class_counts.index],
    autopct='%1.1f%%',
    colors=['#f1c40f', '#95a5a6', '#cd7f32'],
    startangle=90,
    textprops={'fontsize': 11}
)
axes[0].set_title('Passenger Class Distribution', fontweight='bold')

# Right: Bar
axes[1].bar(
    [f'Class {c}' for c in avg_age_class.index],
    avg_age_class.values,
    color=['#f1c40f', '#95a5a6', '#cd7f32'],
    edgecolor='white'
)
axes[1].set_title('Average Age by Class', fontweight='bold')
axes[1].set_xlabel('Passenger Class', fontsize=12)
axes[1].set_ylabel('Average Age (years)', fontsize=12)
axes[1].set_ylim(0, 45)
for i, v in enumerate(avg_age_class.values):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
```
</details>

---

## Summary

In this notebook you learned:

| Chart type | Best for | Key function |
|------------|----------|--------------|
| Bar chart | Comparing categories | `ax.bar()`, `ax.barh()` |
| Histogram | Distribution of one continuous variable | `ax.hist()` |
| Scatter plot | Relationship between two continuous variables | `ax.scatter()` |
| Line chart | Trends over time or ordered groups | `ax.plot()` |
| Pie / Donut | Part-to-whole composition (≤5 groups) | `ax.pie()` |
| Subplots | Multiple charts in one figure | `plt.subplots(r, c)` |

**Core best practices:**
- Always set a **title**, **x-label**, and **y-label**
- Use `plt.tight_layout()` to avoid overlapping elements
- Use `plt.style.use('seaborn-v0_8-whitegrid')` for clean backgrounds
- Encode meaning with color — keep it accessible

---

## Next Steps

- **[3.2 — Advanced Data Visualization](../../3-data-visualization/3.2-adv-data-viz/README.md):** Box plots, violin plots, heatmaps, pair plots, and customizing Matplotlib themes
- **[3.3 — BI with Tableau](../../3-data-visualization/3.3-bi-with-tableau/README.md):** Building interactive dashboards without code
- **[3.4 — Data Storytelling](../../3-data-visualization/3.4-data-storytelling/README.md):** Turning charts into a coherent narrative